# 01. 지도학습 (Supervised Learning)

## 학습 목표
- 회귀와 분류의 차이 이해
- 선형 회귀를 sklearn과 PyTorch로 각각 구현하고 비교
- 로지스틱 회귀와 decision boundary 시각화
- Bias-Variance Tradeoff 직관적 이해
- Regularization (L1, L2)의 효과 비교

## 참고 자료
- [StatQuest - Linear Regression](https://www.youtube.com/watch?v=PaFPbb66DxQ)
- [Stanford CS229 - Supervised Learning](https://cs229.stanford.edu/)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split

np.random.seed(42)
torch.manual_seed(42)

## 1. 지도학습 개요

지도학습은 **입력(X)과 정답(y)** 쌍으로 학습하여, 새로운 입력에 대한 출력을 예측하는 것.

핵심 아이디어: **학습 = 함수 근사(Function Approximation)**

$$\hat{y} = f(X; \theta)$$

- $X$: 입력 데이터
- $\theta$: 학습 가능한 파라미터 (가중치, 편향)
- $\hat{y}$: 예측값

| 문제 유형 | 출력 | 예시 | 손실함수 |
|-----------|------|------|----------|
| 회귀 (Regression) | 연속값 | 집값 예측, 온도 예측 | MSE |
| 분류 (Classification) | 이산값 (카테고리) | 스팸 메일 분류, 숫자 인식 | Cross-Entropy |

In [ ]:
# 회귀 vs 분류 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 회귀: 연속적인 출력
ax = axes[0]
np.random.seed(42)
X_reg = np.linspace(0, 10, 50)
y_reg = 2 * X_reg + 3 + np.random.randn(50) * 2
ax.scatter(X_reg, y_reg, alpha=0.6, s=30)
ax.plot(X_reg, 2 * X_reg + 3, 'r-', linewidth=2, label='$y = 2x + 3$')
ax.set_xlabel('X (예: 면적)')
ax.set_ylabel('y (예: 집값)')
ax.set_title('회귀 (Regression): 연속값 예측')
ax.legend()
ax.grid(True, alpha=0.3)

# 분류: 이산적인 출력
ax = axes[1]
np.random.seed(42)
X_cls, y_cls = make_classification(n_samples=100, n_features=2, n_redundant=0,
                                    n_informative=2, n_clusters_per_class=1, random_state=42)
ax.scatter(X_cls[y_cls==0, 0], X_cls[y_cls==0, 1], alpha=0.6, s=30, label='Class 0', c='blue')
ax.scatter(X_cls[y_cls==1, 0], X_cls[y_cls==1, 1], alpha=0.6, s=30, label='Class 1', c='red')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('분류 (Classification): 카테고리 예측')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. 선형 회귀 (Linear Regression)

가장 기본적인 지도학습 모델. 입력과 출력 사이의 **선형 관계**를 학습.

$$\hat{y} = w \cdot x + b$$

- $w$: 가중치 (weight) - 기울기
- $b$: 편향 (bias) - y절편

### 목표: 예측값과 실제값의 차이(Loss)를 최소화

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

### 2.1 sklearn으로 선형 회귀

In [ ]:
# 데이터 생성
np.random.seed(42)
X = np.linspace(0, 10, 100).reshape(-1, 1)
y = 3 * X.ravel() + 7 + np.random.randn(100) * 2  # y = 3x + 7 + noise

# sklearn 선형 회귀
model_sk = LinearRegression()
model_sk.fit(X, y)

print(f"sklearn 결과:")
print(f"  기울기 (w): {model_sk.coef_[0]:.4f}  (실제: 3.0)")
print(f"  절편 (b):   {model_sk.intercept_:.4f}  (실제: 7.0)")

y_pred_sk = model_sk.predict(X)
mse_sk = np.mean((y - y_pred_sk) ** 2)
print(f"  MSE: {mse_sk:.4f}")

### 2.2 PyTorch로 선형 회귀 (Gradient Descent)

sklearn은 수학적 공식(정규방정식)으로 한 번에 해를 구하지만,
PyTorch는 **경사하강법(Gradient Descent)**으로 반복적으로 파라미터를 업데이트.

이 방식이 딥러닝의 기본 학습 방법이다.

In [ ]:
# PyTorch 텐서로 변환
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)

# 모델 정의
model_pt = nn.Linear(1, 1)  # 입력 1개, 출력 1개

# 손실함수와 옵티마이저
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model_pt.parameters(), lr=0.01)

# 학습 과정 기록
losses = []
w_history = []
b_history = []

# 학습 루프
for epoch in range(200):
    # Forward pass
    y_pred = model_pt(X_tensor)
    loss = criterion(y_pred, y_tensor)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # 기록
    losses.append(loss.item())
    w_history.append(model_pt.weight.item())
    b_history.append(model_pt.bias.item())
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d}: loss={loss.item():.4f}, w={model_pt.weight.item():.4f}, b={model_pt.bias.item():.4f}")

print(f"\nPyTorch 최종 결과:")
print(f"  기울기 (w): {model_pt.weight.item():.4f}  (실제: 3.0)")
print(f"  절편 (b):   {model_pt.bias.item():.4f}  (실제: 7.0)")

### 2.3 sklearn vs PyTorch 결과 비교

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 왼쪽: 데이터 + 두 모델의 예측 비교
ax = axes[0]
ax.scatter(X, y, alpha=0.4, s=15, label='Data')
ax.plot(X, y_pred_sk, 'r-', linewidth=2, label=f'sklearn (w={model_sk.coef_[0]:.2f}, b={model_sk.intercept_:.2f})')
y_pred_pt = model_pt(X_tensor).detach().numpy()
ax.plot(X, y_pred_pt, 'g--', linewidth=2, label=f'PyTorch (w={model_pt.weight.item():.2f}, b={model_pt.bias.item():.2f})')
ax.set_xlabel('X')
ax.set_ylabel('y')
ax.set_title('sklearn vs PyTorch 예측 비교')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 중앙: 학습 과정 (Loss)
ax = axes[1]
ax.plot(losses, 'b-', linewidth=1)
ax.axhline(y=mse_sk, color='r', linestyle='--', label=f'sklearn MSE={mse_sk:.2f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('PyTorch 학습 과정')
ax.legend()
ax.grid(True, alpha=0.3)

# 오른쪽: 파라미터 변화
ax = axes[2]
ax.plot(w_history, label=f'w (최종: {w_history[-1]:.2f})', color='blue')
ax.plot(b_history, label=f'b (최종: {b_history[-1]:.2f})', color='orange')
ax.axhline(y=3.0, color='blue', linestyle='--', alpha=0.5, label='w 실제 (3.0)')
ax.axhline(y=7.0, color='orange', linestyle='--', alpha=0.5, label='b 실제 (7.0)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Parameter Value')
ax.set_title('파라미터 수렴 과정')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.4 Loss Landscape 시각화

경사하강법이 **어떤 공간에서 최적값을 찾아가는지** 시각화.

Loss를 $w$와 $b$의 함수로 그리면 아래와 같은 볼록한(convex) 그릇 모양이 된다.

In [ ]:
# Loss landscape 계산
w_range = np.linspace(-2, 8, 100)
b_range = np.linspace(-5, 20, 100)
W, B = np.meshgrid(w_range, b_range)
Loss_surface = np.zeros_like(W)

for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        y_pred_ij = W[i, j] * X.ravel() + B[i, j]
        Loss_surface[i, j] = np.mean((y - y_pred_ij) ** 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 등고선 + 경사하강법 경로
ax = axes[0]
contour = ax.contour(W, B, Loss_surface, levels=30, cmap='viridis')
ax.clabel(contour, inline=True, fontsize=7)
ax.plot(w_history, b_history, 'r.-', markersize=3, linewidth=1, label='GD path')
ax.plot(w_history[0], b_history[0], 'go', markersize=10, label='Start')
ax.plot(w_history[-1], b_history[-1], 'r*', markersize=15, label='End')
ax.plot(model_sk.coef_[0], model_sk.intercept_, 'bx', markersize=12, markeredgewidth=3, label='Optimal (sklearn)')
ax.set_xlabel('w (기울기)')
ax.set_ylabel('b (절편)')
ax.set_title('Loss Landscape (등고선) + GD 경로')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 오른쪽: 3D 표면
ax = fig.add_subplot(122, projection='3d')
# 기존 axes[1] 제거하고 3D로 교체
axes[1].remove()
ax = fig.add_subplot(122, projection='3d')
ax.plot_surface(W, B, Loss_surface, cmap='viridis', alpha=0.7)
ax.plot(w_history, b_history, losses, 'r.-', markersize=2, linewidth=1)
ax.set_xlabel('w')
ax.set_ylabel('b')
ax.set_zlabel('MSE Loss')
ax.set_title('Loss Surface (3D)')
ax.view_init(elev=30, azim=220)

plt.tight_layout()
plt.show()

---
## 3. 로지스틱 회귀 (Logistic Regression)

분류 문제를 위한 모델. 선형 회귀의 출력을 **시그모이드 함수**에 통과시켜 확률로 변환.

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

$$P(y=1|x) = \sigma(w \cdot x + b)$$

- 출력이 0~1 사이 → 확률로 해석 가능
- 0.5를 기준으로 분류 (threshold)

### 3.1 시그모이드 함수 시각화

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-10, 10, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 왼쪽: 시그모이드 함수
ax = axes[0]
ax.plot(z, sigmoid(z), 'b-', linewidth=2)
ax.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='threshold = 0.5')
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.3)
ax.fill_between(z, sigmoid(z), 0.5, where=(sigmoid(z) >= 0.5), alpha=0.1, color='green', label='Class 1 (y >= 0.5)')
ax.fill_between(z, sigmoid(z), 0.5, where=(sigmoid(z) < 0.5), alpha=0.1, color='red', label='Class 0 (y < 0.5)')
ax.set_xlabel('z = wx + b')
ax.set_ylabel('$\\sigma(z)$')
ax.set_title('Sigmoid Function')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 오른쪽: w 값에 따른 시그모이드 변화
ax = axes[1]
for w in [0.5, 1, 2, 5]:
    ax.plot(z, sigmoid(w * z), label=f'w = {w}')
ax.axhline(y=0.5, color='r', linestyle='--', alpha=0.3)
ax.set_xlabel('x')
ax.set_ylabel('$\\sigma(wx)$')
ax.set_title('w가 클수록 경계가 날카로움')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2 로지스틱 회귀 구현 및 Decision Boundary 시각화

In [ ]:
# 2D 분류 데이터 생성
np.random.seed(42)
X_cls, y_cls = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                    n_informative=2, n_clusters_per_class=1, random_state=42)

# sklearn 로지스틱 회귀
clf = LogisticRegression()
clf.fit(X_cls, y_cls)
print(f"가중치 w: {clf.coef_[0]}")
print(f"절편 b:   {clf.intercept_[0]:.4f}")
print(f"정확도:   {clf.score(X_cls, y_cls):.4f}")

# Decision Boundary 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: Decision Boundary
ax = axes[0]
x_min, x_max = X_cls[:, 0].min() - 1, X_cls[:, 0].max() + 1
y_min, y_max = X_cls[:, 1].min() - 1, X_cls[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                      np.linspace(y_min, y_max, 200))
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
ax.scatter(X_cls[y_cls==0, 0], X_cls[y_cls==0, 1], c='blue', s=20, alpha=0.6, label='Class 0')
ax.scatter(X_cls[y_cls==1, 0], X_cls[y_cls==1, 1], c='red', s=20, alpha=0.6, label='Class 1')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('Decision Boundary (로지스틱 회귀)')
ax.legend()
ax.grid(True, alpha=0.3)

# 오른쪽: 확률 맵
ax = axes[1]
Z_prob = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
contour = ax.contourf(xx, yy, Z_prob, levels=20, cmap='RdBu_r', alpha=0.8)
plt.colorbar(contour, ax=ax, label='P(Class 1)')
ax.contour(xx, yy, Z_prob, levels=[0.5], colors='black', linewidths=2)
ax.scatter(X_cls[y_cls==0, 0], X_cls[y_cls==0, 1], c='blue', s=20, alpha=0.6, edgecolors='white', linewidth=0.5)
ax.scatter(X_cls[y_cls==1, 0], X_cls[y_cls==1, 1], c='red', s=20, alpha=0.6, edgecolors='white', linewidth=0.5)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('확률 맵 (경계에서 0.5, 멀수록 확실)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.3 PyTorch로 로지스틱 회귀 구현

In [ ]:
# PyTorch 데이터 준비
X_t = torch.tensor(X_cls, dtype=torch.float32)
y_t = torch.tensor(y_cls, dtype=torch.float32).reshape(-1, 1)

# PyTorch 모델
class LogisticRegressionPT(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
    
    def forward(self, x):
        return torch.sigmoid(self.linear(x))

model_lr = LogisticRegressionPT(2)
criterion = nn.BCELoss()  # Binary Cross-Entropy
optimizer = torch.optim.SGD(model_lr.parameters(), lr=0.1)

# 학습
losses_lr = []
for epoch in range(300):
    y_pred = model_lr(X_t)
    loss = criterion(y_pred, y_t)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses_lr.append(loss.item())
    if (epoch + 1) % 100 == 0:
        acc = ((y_pred >= 0.5).float() == y_t).float().mean()
        print(f"Epoch {epoch+1}: loss={loss.item():.4f}, accuracy={acc.item():.4f}")

# 최종 정확도
with torch.no_grad():
    y_pred_final = model_lr(X_t)
    acc_final = ((y_pred_final >= 0.5).float() == y_t).float().mean()
    print(f"\nPyTorch 최종 정확도: {acc_final.item():.4f}")
    print(f"sklearn 정확도:      {clf.score(X_cls, y_cls):.4f}")

---
## 4. Bias-Variance Tradeoff

모델의 복잡도를 어떻게 설정하느냐에 따라 두 가지 문제가 발생:

| | 모델이 너무 단순할 때 | 모델이 너무 복잡할 때 |
|---|---|---|
| 이름 | **Underfitting** | **Overfitting** |
| 원인 | High Bias (편향) | High Variance (분산) |
| 증상 | 학습/테스트 모두 성능 나쁨 | 학습 성능 좋지만 테스트 성능 나쁨 |

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

다항 회귀(Polynomial Regression)의 차수를 바꿔가며 이 tradeoff를 직접 확인.

In [ ]:
# 비선형 데이터 생성
np.random.seed(42)
X_poly = np.sort(np.random.uniform(0, 1, 30))
y_poly = np.sin(2 * np.pi * X_poly) + np.random.randn(30) * 0.3  # sin 함수 + 노이즈
X_poly = X_poly.reshape(-1, 1)

X_test_line = np.linspace(0, 1, 200).reshape(-1, 1)
y_true = np.sin(2 * np.pi * X_test_line.ravel())  # 실제 함수

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
degrees = [1, 4, 15]
titles = ['Underfitting\n(Degree 1 - High Bias)',
          'Good Fit\n(Degree 4 - Balanced)',
          'Overfitting\n(Degree 15 - High Variance)']

for ax, degree, title in zip(axes, degrees, titles):
    # 다항 회귀 모델
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_poly, y_poly)
    
    y_pred = model.predict(X_test_line)
    train_mse = np.mean((y_poly - model.predict(X_poly)) ** 2)
    
    ax.scatter(X_poly, y_poly, s=30, alpha=0.7, label='Training data', zorder=5)
    ax.plot(X_test_line, y_true, 'g--', linewidth=1, alpha=0.5, label='True function')
    ax.plot(X_test_line, y_pred, 'r-', linewidth=2, label=f'Degree {degree}')
    ax.set_ylim(-2, 2)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'{title}\nTrain MSE: {train_mse:.4f}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.1 Train/Test MSE vs 모델 복잡도

모델 복잡도(다항식 차수)를 높이면 train error는 계속 줄어들지만, test error는 어느 시점부터 증가.

In [ ]:
# Train/Test 분할
np.random.seed(42)
X_all = np.sort(np.random.uniform(0, 1, 50)).reshape(-1, 1)
y_all = np.sin(2 * np.pi * X_all.ravel()) + np.random.randn(50) * 0.3

X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.3, random_state=42)

degrees = range(1, 16)
train_errors = []
test_errors = []

for d in degrees:
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    model.fit(X_train, y_train)
    
    train_mse = np.mean((y_train - model.predict(X_train)) ** 2)
    test_mse = np.mean((y_test - model.predict(X_test)) ** 2)
    train_errors.append(train_mse)
    test_errors.append(test_mse)

plt.figure(figsize=(8, 5))
plt.plot(degrees, train_errors, 'b-o', label='Train MSE', markersize=5)
plt.plot(degrees, test_errors, 'r-o', label='Test MSE', markersize=5)
plt.axvline(x=4, color='green', linestyle='--', alpha=0.5, label='Sweet spot (degree~4)')
plt.xlabel('Polynomial Degree (모델 복잡도)')
plt.ylabel('MSE')
plt.title('Bias-Variance Tradeoff: Train vs Test Error')
plt.legend()
plt.ylim(0, min(max(test_errors), 5))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"최적 차수: {degrees[np.argmin(test_errors)]} (Test MSE: {min(test_errors):.4f})")

---
## 5. Regularization (정규화)

Overfitting을 방지하기 위해 **가중치의 크기에 페널티**를 부여.

$$\text{Loss}_{\text{regularized}} = \text{Loss}_{\text{original}} + \lambda \cdot \text{Penalty}$$

| 방법 | 페널티 | 효과 | sklearn |
|------|--------|------|----------|
| L1 (Lasso) | $\lambda \sum |w_i|$ | 불필요한 가중치를 **0으로** (Feature Selection) | `Lasso` |
| L2 (Ridge) | $\lambda \sum w_i^2$ | 가중치를 **전반적으로 작게** | `Ridge` |

- $\lambda$ (alpha): 정규화 강도. 클수록 더 강한 페널티.

### 5.1 Overfitting에 대한 Regularization 효과

In [ ]:
# 높은 차수 다항 회귀에 정규화 적용
degree = 15  # 정규화 없으면 심하게 오버피팅

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models = [
    ('No Regularization', make_pipeline(PolynomialFeatures(degree), LinearRegression())),
    ('L2 (Ridge, alpha=0.01)', make_pipeline(PolynomialFeatures(degree), Ridge(alpha=0.01))),
    ('L1 (Lasso, alpha=0.01)', make_pipeline(PolynomialFeatures(degree), Lasso(alpha=0.01, max_iter=10000))),
]

X_plot = np.linspace(0, 1, 200).reshape(-1, 1)
y_true_plot = np.sin(2 * np.pi * X_plot.ravel())

for ax, (name, model) in zip(axes, models):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_plot)
    
    train_mse = np.mean((y_train - model.predict(X_train)) ** 2)
    test_mse = np.mean((y_test - model.predict(X_test)) ** 2)
    
    ax.scatter(X_train, y_train, s=30, alpha=0.7, label='Train', zorder=5)
    ax.scatter(X_test, y_test, s=30, alpha=0.7, marker='^', label='Test', zorder=5)
    ax.plot(X_plot, y_true_plot, 'g--', alpha=0.5, label='True')
    ax.plot(X_plot, y_pred, 'r-', linewidth=2, label='Prediction')
    ax.set_ylim(-2, 2)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'{name}\nTrain MSE: {train_mse:.4f}, Test MSE: {test_mse:.4f}')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.2 L1 vs L2: 가중치 비교

L1은 불필요한 가중치를 0으로 만들고 (희소성, Sparsity),
L2는 모든 가중치를 고르게 줄인다.

In [ ]:
# 가중치 비교
degree = 10

# 세 모델 학습
model_none = make_pipeline(PolynomialFeatures(degree), LinearRegression())
model_ridge = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=1.0))
model_lasso = make_pipeline(PolynomialFeatures(degree), Lasso(alpha=0.01, max_iter=10000))

model_none.fit(X_train, y_train)
model_ridge.fit(X_train, y_train)
model_lasso.fit(X_train, y_train)

# 가중치 추출
w_none = model_none.named_steps['linearregression'].coef_
w_ridge = model_ridge.named_steps['ridge'].coef_
w_lasso = model_lasso.named_steps['lasso'].coef_

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, w) in zip(axes, [('No Regularization', w_none),
                                  ('L2 (Ridge)', w_ridge),
                                  ('L1 (Lasso)', w_lasso)]):
    colors = ['red' if abs(wi) < 0.01 else 'steelblue' for wi in w]
    ax.bar(range(len(w)), w, color=colors)
    ax.set_xlabel('Feature index')
    ax.set_ylabel('Weight value')
    ax.set_title(f'{name}\n(0에 가까운 가중치: {sum(abs(wi) < 0.01 for wi in w)}/{len(w)})')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"정규화 없음 - 가중치 범위: [{w_none.min():.1f}, {w_none.max():.1f}]")
print(f"Ridge (L2) - 가중치 범위: [{w_ridge.min():.1f}, {w_ridge.max():.1f}]")
print(f"Lasso (L1) - 가중치 범위: [{w_lasso.min():.1f}, {w_lasso.max():.1f}]")
print(f"Lasso (L1) - 0인 가중치: {sum(abs(wi) < 0.01 for wi in w_lasso)}/{len(w_lasso)} → Feature Selection 효과")

### 5.3 정규화 강도(alpha) 변화에 따른 효과

In [ ]:
alphas = np.logspace(-4, 2, 50)
ridge_train = []
ridge_test = []
lasso_train = []
lasso_test = []

degree = 10

for alpha in alphas:
    # Ridge
    m = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=alpha))
    m.fit(X_train, y_train)
    ridge_train.append(np.mean((y_train - m.predict(X_train)) ** 2))
    ridge_test.append(np.mean((y_test - m.predict(X_test)) ** 2))
    
    # Lasso
    m = make_pipeline(PolynomialFeatures(degree), Lasso(alpha=alpha, max_iter=10000))
    m.fit(X_train, y_train)
    lasso_train.append(np.mean((y_train - m.predict(X_train)) ** 2))
    lasso_test.append(np.mean((y_test - m.predict(X_test)) ** 2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogx(alphas, ridge_train, 'b-', label='Train')
ax.semilogx(alphas, ridge_test, 'r-', label='Test')
best_alpha = alphas[np.argmin(ridge_test)]
ax.axvline(x=best_alpha, color='green', linestyle='--', alpha=0.5, label=f'Best alpha={best_alpha:.4f}')
ax.set_xlabel('alpha (정규화 강도)')
ax.set_ylabel('MSE')
ax.set_title('Ridge (L2) - alpha에 따른 Train/Test Error')
ax.legend()
ax.set_ylim(0, min(max(ridge_test), 3))
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.semilogx(alphas, lasso_train, 'b-', label='Train')
ax.semilogx(alphas, lasso_test, 'r-', label='Test')
best_alpha = alphas[np.argmin(lasso_test)]
ax.axvline(x=best_alpha, color='green', linestyle='--', alpha=0.5, label=f'Best alpha={best_alpha:.4f}')
ax.set_xlabel('alpha (정규화 강도)')
ax.set_ylabel('MSE')
ax.set_title('Lasso (L1) - alpha에 따른 Train/Test Error')
ax.legend()
ax.set_ylim(0, min(max(lasso_test), 3))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 다중 선형 회귀 (sklearn vs PyTorch)

Boston Housing과 유사한 데이터를 생성하여 다중 선형 회귀를 구현하세요.
sklearn과 PyTorch 두 가지로 구현하고 결과를 비교하세요.

In [ ]:
from sklearn.datasets import make_regression

# 다중 회귀 데이터: 5개 특성, 100개 샘플
X_multi, y_multi = make_regression(n_samples=100, n_features=5, noise=10, random_state=42)

# TODO: sklearn LinearRegression으로 학습하고 가중치, R² score 출력

# TODO: PyTorch nn.Linear(5, 1)로 같은 데이터 학습
#   - MSELoss, Adam optimizer (lr=0.01) 사용
#   - 200 epoch 학습 후 loss 변화 그래프 그리기
#   - 최종 가중치를 sklearn과 비교


### 연습 2: Decision Boundary 시각화

make_moons 데이터셋에 대해 로지스틱 회귀와 다항 특성을 사용한 로지스틱 회귀의 decision boundary를 비교하세요.

In [ ]:
from sklearn.datasets import make_moons

# 반달 모양 데이터 (비선형 경계 필요)
X_moon, y_moon = make_moons(n_samples=200, noise=0.2, random_state=42)

# TODO: 두 가지 모델의 decision boundary를 나란히 시각화
#   1. 기본 로지스틱 회귀 (선형 경계)
#   2. PolynomialFeatures(degree=3) + 로지스틱 회귀 (비선형 경계)
#   각각의 정확도도 출력하세요


### 연습 3: 최적의 정규화 강도 찾기

Cross-validation을 사용하여 Ridge 회귀의 최적 alpha를 찾으세요.

In [ ]:
from sklearn.model_selection import cross_val_score

# 데이터 생성
np.random.seed(42)
X_cv = np.sort(np.random.uniform(0, 1, 100)).reshape(-1, 1)
y_cv = np.sin(2 * np.pi * X_cv.ravel()) + np.random.randn(100) * 0.3

# TODO: alpha를 10^(-4) ~ 10^(2) 범위에서 20개 탐색
#   - 각 alpha에 대해 PolynomialFeatures(degree=10) + Ridge(alpha)
#   - 5-fold cross_val_score로 평균 MSE 계산 (scoring='neg_mean_squared_error')
#   - alpha vs CV MSE 그래프 그리기
#   - 최적 alpha 출력


---
## 핵심 정리

| 개념 | 핵심 내용 | ML/DL에서의 역할 |
|------|-----------|-------------------|
| 지도학습 | 입력-정답 쌍으로 함수 근사 | 모든 ML의 기본 프레임워크 |
| 선형 회귀 | $\hat{y} = wx + b$, MSE 최소화 | 가장 단순한 모델, 딥러닝 레이어의 기본 |
| 로지스틱 회귀 | 시그모이드로 확률 출력 | 분류의 기본, 뉴런의 활성화 함수 |
| Bias-Variance | 모델 복잡도의 균형점 찾기 | 모델 선택의 핵심 기준 |
| L1 정규화 (Lasso) | 가중치를 0으로 (Feature Selection) | 희소 모델, Pruning |
| L2 정규화 (Ridge) | 가중치를 작게 (Weight Decay) | 딥러닝의 weight decay |
| sklearn vs PyTorch | 닫힌 해 vs 경사하강법 | sklearn: 빠른 실험, PyTorch: 유연한 모델 |

**다음 노트북**: [02-loss-and-optimization.ipynb](02-loss-and-optimization.ipynb) - 손실함수와 최적화